# `from_sensorfilter` — the PSF follows the filter

`Simulation.from_sensorfilter` and `ImageSimulator.from_sensorfilter` build from a
canonical **sensorfilter label** (`kind:band`, e.g. `'zwo:r'`). Unlike
`from_sensor_and_scene` they do two things for you:

1. pick the throughput curve for that sensor/filter combination, and
2. pick the PSF from the filter's `focus_level`, exposed as `default_psf`.

| `focus_level` | PSF | example label |
|---|---|---|
| `0wave` (in focus) | `AiryPSF` | `zwo:r`, `zwo:bb` |
| `1wave` (±1 wave defocus) | `DefocusPSF` (1 wave) | `zwo:r+1`, `zwo:r-1` |
| `2wave` (+2 wave defocus) | `DefocusPSF` (2 wave) | `zwo:bb2` |

Because the PSF rides along on the simulation, `simulate()`, `get_image_snr()`, and
`get_image_exptime_for_snr()` all use it **without a `psf=` argument**.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

import wcc_etc

wcc_etc.set_wcc_style()

scene = wcc_etc.get_scene(
    name="G5V",
    mag=19,
    background="zodi",
    bandpass="johnson_r",
    background_prop={"bandpass": "johnson_r", "mag": 22.5},
)

LABELS = ["zwo:r", "zwo:r+1", "zwo:bb2"]

## 1. Label to PSF

The `+1` / `-1` / `2` suffixes mark a defocused sensor. `default_psf` reports what was
selected.

In [ ]:
for label in LABELS:
    sim = wcc_etc.Simulation.from_sensorfilter(label, scene)
    print(f"{label:<10}{type(sim.default_psf).__name__}")

## 2. What the auto-selected PSFs look like

`ImageSimulator.from_sensorfilter(...).simulate(time)` renders a bright point source with
the filter's own PSF — again, no `psf=` argument. The in-focus image is a tight Airy core;
the 1-wave and 2-wave images spread the light into progressively larger doughnuts.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4.4))
for ax, label in zip(axes, LABELS):
    imsim = wcc_etc.ImageSimulator.from_sensorfilter(
        label, scene, npix=160, oversample=5
    )
    res = imsim.simulate(time=5, add_noise=False)
    res.plot_image(
        ax=ax,
        noise=False,
        stretch="log",
        cmap="magma",
        colorbar=False,
        title=f"{label}\n{type(imsim.default_psf).__name__}",
    )

fig.suptitle("Auto-selected PSF per focus level (log stretch)", y=1.02)
fig.tight_layout()
plt.show()

## 3. Defocus costs SNR

Spreading the same photons over more pixels means more noise pixels inside a fixed
aperture, so at any exposure time the defocused labels sit below the in-focus one. No
`psf=` is passed anywhere here.

At 60 s on this r = 19 source that costs a factor of **3.6** for one wave of defocus
(SNR **192.7 → 53.5**) and **4.8** for two (SNR **40.3**) — at the default aperture.
`04_psf_and_image_snr.ipynb` shows how much of that comes back when the aperture is
matched to the broader PSF.

In [ ]:
times = np.logspace(0, 2.2, 40)

fig, ax = plt.subplots(figsize=(7, 4.2))
for label in LABELS:
    sim = wcc_etc.Simulation.from_sensorfilter(label, scene)
    snr = [sim.get_image_snr(time=t)["snr"] for t in times]
    ax.loglog(times, snr, label=f"{label}  ({type(sim.default_psf).__name__})")

ax.set_xlabel("Exposure time [s]")
ax.set_ylabel("SNR")
ax.set_title("SNR vs exposure time by focus level (G5V, r = 19)")
ax.legend(fontsize=9)
plt.show()

## 4. Guards

`from_sensorfilter` accepts only **canonical** labels that have a throughput curve. An
unknown label raises `ValueError`; a known-but-unimplemented one (a narrowband filter with
no curve yet) raises `NotImplementedError`. Nicknames like `'sony:r'` are not canonical —
use `from_sensor_and_scene` for those.

In [ ]:
for label in ["zwo:does_not_exist", "zwo:halpha"]:
    try:
        wcc_etc.Simulation.from_sensorfilter(label, scene)
    except (ValueError, NotImplementedError) as err:
        print(f"{label:<20}{type(err).__name__}: {err}")

## Summary

- `Simulation.from_sensorfilter(label, scene)` and
  `ImageSimulator.from_sensorfilter(label, scene, npix=..., oversample=...)` build from a
  canonical `kind:band` label.
- The PSF comes from the filter's `focus_level` and is exposed as `default_psf`
  (`AiryPSF` for `0wave`, `DefocusPSF` for `1wave` / `2wave`).
- `simulate()`, `get_image_snr()`, and `get_image_exptime_for_snr()` all use it when no
  `psf=` is given, and defocus lowers the SNR at a fixed aperture.
- Unknown labels raise `ValueError`; unimplemented ones raise `NotImplementedError`.